# UFC dataset and fight models

This notebook runs the whole pipeline and shows its products:

1. **The dataset**: every UFC fight since 1993, with round-by-round statistics, judges' scores, bonuses, betting odds, official rankings and professional records, written to `dataset/`.
2. **Fight-outcome models**: how well can the result of a UFC fight be predicted from what was known before it?
3. **Division rankings**: the active fighters of each division, ranked by three independent methods.

Each step writes its output to `data/processed/` (and the dataset to `dataset/`). The three analysis notebooks read those files, so run this one first:

| Notebook | Content |
|---|---|
| [01_exploration](notebooks/01_exploration.ipynb) | The data: coverage, the corner artefact, how fights end, rounds, judges, the betting market, official rankings, professional records |
| [02_feature_engineering](notebooks/02_feature_engineering.ipynb) | How every feature is built without leaking the future, how much signal each one carries, and whether the new data help |
| [03_models_and_rankings](notebooks/03_models_and_rankings.ipynb) | Model diagnostics, Elo tuning, and a critical look at the rankings |

The same pipeline runs from the command line with `python -m ufc_rating.pipeline`.

## 0. Configuration

In [1]:
# Data
REFRESH_DATA = False      # True: refresh Kaggle, Wikipedia and bestfightodds first (network; Kaggle API credentials)
SCRAPE_UFCSTATS = False   # True: also scrape ufcstats.com for events newer than the data (needs the 'scrape' extra)

# Weighted ranking: weight of each career statistic (must sum to 1)
WEIGHTS = {
    'win_rate':    0.20,   # wins / decided fights
    'finish_rate': 0.15,   # wins by KO/TKO or submission
    'slpm':        0.15,   # significant strikes landed per minute
    'sig_acc':     0.10,   # significant strike accuracy
    'td_per15':    0.10,   # takedowns per 15 minutes
    'td_acc':      0.10,   # takedown accuracy
    'ctrl_pct':    0.10,   # share of fight time in control
    'kd_per15':    0.05,   # knockdowns per 15 minutes
    'sub_per15':   0.05,   # submission attempts per 15 minutes
}

# Who gets ranked
ACTIVE_DAYS = 730   # fought in the last two years
MIN_FIGHTS = 5      # at least five UFC fights
TOP_N = 10          # fighters shown per division

In [2]:
import pandas as pd
from IPython.display import Markdown, display

from ufc_rating.config import DATASET_DIR, DIVISIONS
from ufc_rating.dataset import export
from ufc_rating.models.training import format_scores
from ufc_rating.pipeline import build_features, compare_feature_groups, fit_models, rank_fighters, update_data

pd.set_option('display.width', 140)

## 1. Data

| Source | Content |
|---|---|
| Kaggle mirror of [ufcstats.com](http://ufcstats.com) (CC0) | Results, fight totals, round-by-round statistics, judges' scores, bonuses, fighter profiles. Only UFC events are kept. |
| Our ufcstats.com scraper | The events more recent than the mirror, in the same format (`data/raw/scraped/`) |
| Ultimate UFC Dataset (Kaggle, CC BY 4.0) | Betting odds and official ranks at fight time, 2010 to March 2026 |
| [bestfightodds.com](https://www.bestfightodds.com) | Closing odds of the later events (median over the sportsbooks) |
| English Wikipedia (CC BY-SA 4.0) | The official rankings week by week since 2018, and the professional records of the fighters |

Odds and rankings are matched to the fights on fighter names and dates; details in [01_exploration](notebooks/01_exploration.ipynb).

In [3]:
master, rounds = update_data(refresh=REFRESH_DATA, scrape=SCRAPE_UFCSTATS)

Master table: 8,905 UFC fights, 1993-11-12 to 2026-09-19; 20,904 rounds with stats


## 2. Features

Every decided fight becomes a comparison between two fighters **A** and **B**, drawn at random from the two corners, described by the difference of their career statistics, physical attributes and Elo rating **before** the fight. Draws, no contests and fights involving a UFC debutant (no history to learn from) are left out.

The Elo rating is updated after every fight (K = 80, split and majority decisions counting half). Why the corners have to be randomised, how each feature avoids leaking the future, and the round, judges and bonus features that were tested but not kept: [02_feature_engineering](notebooks/02_feature_engineering.ipynb).

In [4]:
elo_history, matchups, profiles = build_features(master, rounds)
ablation = compare_feature_groups(matchups)   # written to data/processed/, shown in notebook 02

Matchups: 6,525 fights with two experienced fighters, 24 stat features (+ odds)
Feature groups (rolling-origin log loss before the test period):
                             mean log loss  vs first set  folds better
features                                                              
model features                      0.6584        0.0000             0
+ rounds                            0.6583       -0.0002             2
+ judges                            0.6584       -0.0000             3
+ bonuses                           0.6579       -0.0005             4
+ rounds + judges + bonuses         0.6577       -0.0007             2


## 3. The dataset

The clean tables published in `dataset/`, with a generated dataset card (`dataset/README.md`) describing every file, its coverage and its sources.

In [5]:
tables = export(master, rounds, elo_history)
pd.DataFrame({'rows': {name: len(t) for name, t in tables.items()},
              'columns': {name: t.shape[1] for name, t in tables.items()}})

,rows,columns
fights,8905,84
rounds,20904,50
fighters,2760,19
elo,17810,6
rankings,77503,7
records,46223,12


In [6]:
tables['fights'].tail(3).T.head(40)

,8902,8903,8904
fight_id,d89cc2cb92cd8407,f7810c98786c3dd8,fe612d20eec7ef35
event_id,8a0a35e7c74bebcc,8a0a35e7c74bebcc,8a0a35e7c74bebcc
event,UFC 331: Van vs. Pantoja 2,UFC 331: Van vs. Pantoja 2,UFC 331: Van vs. Pantoja 2
date,2026-09-19,2026-09-19,2026-09-19
location,"Los Angeles, California, USA","Los Angeles, California, USA","Los Angeles, California, USA"
division,Featherweight,Featherweight,Lightweight
weight_class,Featherweight,Featherweight,Lightweight
title_fight,0,0,0
scheduled_rounds,3,3,5
time_format,3 Rnd (5-5-5),3 Rnd (5-5-5),5 Rnd (5-5-5-5-5)


## 4. Models

The fights are split in time: the oldest 70% for training, the next 15% for validation, the most recent 15% for the final test. Four models (logistic regression, SVM, random forest, XGBoost) are tuned by time-ordered cross-validation on the training period, in two versions:

- **stats**: fighter data only. This is the model used for the rankings.
- **stats + odds**: the same inputs plus the bookmakers' implied probability.

Validation picks the stats model used for the rankings; the test period is scored once, at the end. For the rankings, which describe the fighters of today, the selected model is then refitted on every fight.

In [7]:
fitted = fit_models(matchups)
results = fitted['results']

pd.DataFrame(results['split'], index=['from', 'to', 'fights']).T

Stats-only models:


  LogReg        CV log-loss 0.6607  {'C': 0.01}


  SVM           CV log-loss 0.6608  {'C': 0.01, 'kernel': 'linear'}


  RandomForest  CV log-loss 0.6681  {'max_depth': 8, 'min_samples_leaf': 10}


  XGBoost       CV log-loss 0.6691  {'learning_rate': 0.02, 'max_depth': 2}
Stats + odds models:
  LogReg        CV log-loss 0.6328  {'C': 0.01}


  SVM           CV log-loss 0.6365  {'C': 0.01, 'kernel': 'linear'}


  RandomForest  CV log-loss 0.6474  {'max_depth': 8, 'min_samples_leaf': 10}


  XGBoost       CV log-loss 0.6430  {'learning_rate': 0.02, 'max_depth': 2}


,from,to,fights
train,1994-03-11 00:00:00,2022-01-22 00:00:00,4565
validation,2022-02-05 00:00:00,2024-06-01 00:00:00,978
test,2024-06-08 00:00:00,2026-09-19 00:00:00,982


**Test period, all fights.** Accuracy is the share of winners correctly predicted (50% is a coin flip, since A is drawn at random). AUC measures how well the predicted probabilities rank winners above losers (0.5 = chance). Log loss and Brier score measure the quality of the probabilities themselves (lower is better). *Elo only* predicts with the pre-fight Elo ratings alone.

In [8]:
format_scores(results['test_stats'])

,Fights,Accuracy,AUC,Log loss,Brier
LogReg,982,65.8%,0.715,0.626,0.218
SVM,982,66.0%,0.715,0.625,0.217
RandomForest,982,65.1%,0.702,0.640,0.224
XGBoost,982,64.7%,0.706,0.634,0.222
Elo only,982,56.1%,0.591,0.681,0.244


**Against the betting market.** Only the test fights with odds, so every line is scored on exactly the same fights.

In [9]:
format_scores(results['test_market'])

,Fights,Accuracy,AUC,Log loss,Brier
Betting favourite (market),911,69.9%,0.758,0.586,0.200
Elo only,911,56.5%,0.599,0.678,0.243
LogReg (stats),911,66.0%,0.715,0.626,0.218
SVM (stats),911,66.2%,0.715,0.625,0.217
RandomForest (stats),911,65.3%,0.703,0.639,0.224
XGBoost (stats),911,65.1%,0.707,0.633,0.221
LogReg (stats + odds),911,69.5%,0.758,0.587,0.200
SVM (stats + odds),911,69.3%,0.757,0.588,0.201
RandomForest (stats + odds),911,68.6%,0.751,0.601,0.206
XGBoost (stats + odds),911,69.0%,0.756,0.589,0.202


Reading these two tables:

- Fighter statistics alone predict the winner clearly better than chance and better than Elo alone.
- The betting market stays the reference: bookmakers see everything the statistics see, plus injuries, camps, weight cuts and style match-ups.
- Adding the statistics to the odds brings the models level with the market, not above it: the public statistics carry no information the market has not already priced in.

Calibration, confidence and what each model relies on: [03_models_and_rankings](notebooks/03_models_and_rankings.ipynb).

## 5. Rankings

Active fighters (a fight in the last `ACTIVE_DAYS` days, at least `MIN_FIGHTS` UFC fights) are ranked in their most recent division by three methods:

- **Elo**: dynamic rating updated after every fight since 1993 (win = 1, draw = 0.5, K = 80, split and majority decisions counting half). It rewards *who* you beat.
- **Weighted**: career statistics turned into percentiles within the division and combined with the `WEIGHTS` above. It rewards *how* you fight.
- **Model**: a virtual round-robin tournament. The selected stats model, refitted on all fights, predicts every possible match-up in the division; the score is the fighter's average win probability.

Fighters are sorted by the **consensus**, the mean of the three ranks.

In [10]:
as_of = master['date'].max()
best = results['best_stats_model']
rankings = rank_fighters(profiles, fitted['ranking_model'], as_of, WEIGHTS,
                         active_days=ACTIVE_DAYS, min_fights=MIN_FIGHTS)
print(f'Rankings as of {as_of.date()}, round-robin model: {best}')

Rankings as of 2026-09-19, round-robin model: LogReg


In [11]:
columns = ['Fighter', 'UFC record', 'Last fight', 'Elo rank', 'Weighted rank', 'Model rank', 'Consensus']
for division in DIVISIONS:
    table = rankings[rankings['division'] == division]
    if table.empty:
        continue
    display(Markdown(f'### {division} ({len(table)} active fighters)'))
    display(table.set_index('rank')[columns].head(TOP_N))

### Flyweight (35 active fighters)

,Fighter,UFC record,Last fight,Elo rank,Weighted rank,Model rank,Consensus
rank,,,,,,,
1,Joshua Van,11-1,2026-09-19,1,3,1,1.7
2,Tatsuro Taira,8-2,2026-05-09,4,2,2,2.7
3,Manel Kape,8-3,2026-06-20,2,6,3,3.7
4,Alexandre Pantoja,14-5,2026-09-19,3,1,8,4.0
5,Asu Almabayev,7-1,2026-06-27,6,4,4,4.7
6,Kyoji Horiguchi,9-2,2026-06-20,5,8,9,7.3
7,Andre Lima,5-1,2026-08-29,14,5,6,8.3
8,Brandon Moreno,12-7-2,2026-09-12,7,14,7,9.3
9,Tagir Ulanbekov,6-2,2025-11-22,8,9,13,10.0


### Bantamweight (53 active fighters)

,Fighter,UFC record,Last fight,Elo rank,Weighted rank,Model rank,Consensus
rank,,,,,,,
1,Sean O'Malley,12-3,2026-06-14,3,4,4,3.7
2,Umar Nurmagomedov,8-2,2026-08-29,6,6,2,4.7
3,Petr Yan,12-4,2025-12-06,2,7,5,4.7
4,Merab Dvalishvili,14-3,2025-12-06,1,14,1,5.3
5,Mario Bautista,12-3,2026-07-11,5,2,9,5.3
6,Montel Jackson,9-4,2026-04-25,12,8,6,8.7
7,Farid Basharat,7-0,2026-07-11,7,12,7,8.7
8,Song Yadong,13-4-1,2026-08-29,4,16,8,9.3
9,Raul Rosas Jr.,6-1,2026-03-07,17,9,3,9.7


### Featherweight (56 active fighters)

...

,Fighter,UFC record,Last fight,Elo rank,Weighted rank,Model rank,Consensus
rank,,,,,,,
1,Tatiana Suarez,9-1,2026-04-11,1,2,1,1.3
2,Fatima Kline,4-1,2026-07-18,6,1,2,3.0
3,Denise Gomes,7-2,2026-08-29,4,3,4,3.7
4,Gillian Robertson,14-7,2026-08-15,5,8,6,6.3
5,Virna Jandiroba,9-4,2026-04-04,3,10,7,6.7
6,Jaqueline Amorim,5-2,2026-05-30,10,7,5,7.3
7,Loopy Godinez,9-6,2026-04-11,13,5,8,8.7
8,Mackenzie Dern,12-5,2026-08-15,2,15,9,8.7
9,Iasmin Lucindo,5-2,2025-08-09,9,17,3,9.7


### Women's Flyweight (30 active fighters)

,Fighter,UFC record,Last fight,Elo rank,Weighted rank,Model rank,Consensus
rank,,,,,,,
1,Valentina Shevchenko,15-3-1,2025-11-15,1,3,3,2.3
2,Erin Blanchfield,8-1,2025-11-15,4,4,2,3.3
3,Zhang Weili,10-3,2025-11-15,3,1,8,4.0
4,Natalia Silva,8-0,2026-01-24,2,11,1,4.7
5,Casey O'Neill,7-2,2026-09-19,11,2,6,6.3
6,Maycee Barber,10-3,2026-03-28,9,6,5,6.7
7,Carli Judice,4-1,2026-08-22,13,5,4,7.3
8,Wang Cong,5-1,2026-07-11,10,7,10,9.0
9,Manon Fiorot,8-2,2026-09-12,6,14,9,9.7


### Women's Bantamweight (20 active fighters)

,Fighter,UFC record,Last fight,Elo rank,Weighted rank,Model rank,Consensus
rank,,,,,,,
1,Luana Santos,6-1,2026-06-20,6,1,1,2.7
2,Ailin Perez,6-1,2026-02-28,4,2,2,2.7
3,Joselyne Edwards,9-4,2026-04-25,5,3,4,4.0
4,Norma Dumont,9-3,2026-04-25,2,7,5,4.7
5,Raquel Pennington,13-6,2024-10-05,1,10,6,5.7
6,Julianna Pena,8-4,2025-06-07,7,4,10,7.0
7,Karol Rosa,8-5,2026-06-20,11,6,7,8.0
8,Ketlen Vieira,10-5,2026-05-16,3,15,9,9.0
9,Jacqueline Cavalcanti,5-1,2026-05-16,9,16,3,9.3


How far the three methods agree, how they compare with the official UFC rankings (the media panel and the new Meta UFC Rankings), and the all-time Elo table: [03_models_and_rankings](notebooks/03_models_and_rankings.ipynb).